# Amaliy mashgʻulot 15. Dialog tizimlari: intentlar, slotlar, ssenariylar va generativ modellar

15-laboratoriyada siz foydalanuvchi gapi qanday qilib intent va slotlarga aylanishini, freym dialog davomida qanday toʻldirilishini va ssenariyli tizim generativ tizimdan nimasi bilan farq qilishini koʻrgansiz. Bu yerda shunday tizimni kodda yigʻamiz (soha: poyezd chiptasi sotib olish va restoranda stol band qilish):

1. intentlar klassifikatori: numpy da noldan softmax-regressiya, keyin scikit-learn;
2. BIO belgilashda slotlarni toʻldirish: qoidalar va CRF (`sklearn-crfsuite`);
3. freymli dialog menejeri (chekli avtomat) va shablon asosida javob generatsiyasi;
4. oʻyinchoq generativ javob modeli: nega u slot qiymatlarini «toʻqib chiqaradi» va deleksikalizatsiya qanday yordam beradi.

*«NLP algoritmlari» kursining 15-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Maʼlumotlar: belgilangan gaplar

Har bir gap intent (foydalanuvchi nima istaydi) va slotlar (soʻrov parametrlari) bilan belgilangan. Slotlar matnning oʻzida `[qiymat](slot)` koʻrinishida yozilgan — chat-botlar uchun koʻp freymvorklarda maʼlumotlar shunday belgilanadi. `parse_markup` funksiyasi belgilashni tokenlar va BIO teglariga aylantiradi: `B-` — slotning birinchi soʻzi, `I-` — davomi, `O` — slotdan tashqari.

Jami 60 ta oʻquv gapi (6 ta intent, har biriga 10 tadan) va 12 ta test gapi. Oʻz maʼlumotlaringizni sinamoqchi boʻlsangiz, faylni «Files» paneli orqali yuklang va uni `train_data` ga oʻqing.

In [ ]:
!pip install -q sklearn-crfsuite

In [ ]:
# Slotlar belgisi toʻgʻridan-toʻgʻri matnda: [qiymat](slot). Slotlar: from, to, date, time, count
train_data = {
    "buy_ticket": [
        "[Toshkentdan](from) [Samarqandga](to) chipta olmoqchiman",
        "[Ertaga](date) [Buxoroga](to) chipta kerak",
        "[Juma kuni](date) [Samarqandga](to) [ikkita](count) chipta oling",
        "[Samarqanddan](from) [Toshkentga](to) [10-may](date) uchun chipta bering",
        "[Xivaga](to) poyezdga [uchta](count) joy band qiling",
        "[Indinga](date) [ertalab](time) [Buxoroga](to) ketmoqchiman",
        "[Shanba kuni](date) [Buxorodan](from) [Toshkentga](to) chipta bormi?",
        "[Samarqandgacha](to) Afrosiyob poyezdiga chipta rasmiylashtiring",
        "[Ertaga](date) [ertalabki](time) poyezdga [ikkita](count) chipta kerak",
        "[Toshkent](from) – [Nukus](to) chiptasini sotib olmoqchiman",
    ],
    "schedule": [
        "[Samarqandga](to) poyezd qachon joʻnaydi?",
        "[Toshkentdan](from) [Buxoroga](to) poyezd soat nechada?",
        "[Ertaga](date) [Xivaga](to) poyezdlar jadvali qanday?",
        "Afrosiyob qachon joʻnab ketadi?",
        "[Kechqurun](time) [Qarshiga](to) poyezd bormi?",
        "[Buxorogacha](to) poyezd necha soat yuradi?",
        "[Samarqanddan](from) eng yaqin poyezd qachon?",
        "[Juma kungi](date) jadvalni koʻrsating",
        "Poyezd [Toshkentga](to) soat nechada yetib keladi?",
        "[Ertalab](time) [Nukusga](to) qanday poyezdlar bor?",
    ],
    "book_table": [
        "[Toʻrt](count) kishilik stol band qilmoqchiman",
        "[Bugun](date) soat [19:00](time) ga stol band qiling",
        "[Ertaga](date) [kechqurun](time) [ikki](count) kishilik stol kerak",
        "[Shanba kuni](date) restoranda stol buyurtma qilsa boʻladimi?",
        "[Besh](count) kishi uchun stol band qiling",
        "[Bugun](date) soat [20:00](time) da sizlarda kechki ovqat qilmoqchimiz",
        "Soat [18:30](time) ga [uch](count) kishilik boʻsh stol bormi?",
        "Bizni [juma kuni](date) kechki ovqatga yozib qoʻying",
        "Iltimos, deraza yonidagi stolni band qiling",
        "[Ertaga](date) [olti](count) kishilik stol kerak",
    ],
    "cancel": [
        "Buyurtmamni bekor qiling",
        "Bronni bekor qilmoqchiman",
        "Bekor qiling, fikrimni oʻzgartirdim",
        "Stol band qilish shart emas",
        "Chipta pulini qaytaring",
        "Chiptani qaytarmoqchiman",
        "Stol bronini bekor qilish",
        "Safarni bekor qilaman",
        "Iltimos, buyurtmani bekor qiling",
        "Rejalar oʻzgardi, hammasini bekor qiling",
    ],
    "greeting": [
        "Assalomu alaykum",
        "Salom",
        "Xayrli kun",
        "Xayrli tong",
        "Xayrli kech",
        "Assalomu alaykum, menga yordam kerak",
        "Salom, bot",
        "Allo, eshityapsizmi?",
        "Salomlar",
        "Assalomu alaykum, bir savol bersam boʻladimi?",
    ],
    "thanks_bye": [
        "Rahmat",
        "Rahmat, xayr",
        "Yordamingiz uchun katta rahmat",
        "Boʻldi, xayr",
        "Xayr",
        "Tashakkur",
        "Rahmat, shu xolos",
        "Xayr, salomat boʻling",
        "Hammasi uchun rahmat",
        "Zoʻr, rahmat",
    ],
}

test_data = [
    ("[Toshkentdan](from) [Xivaga](to) chipta kerak", "buy_ticket"),
    ("[Dushanba kuni](date) [Samarqandgacha](to) [bitta](count) chipta oling", "buy_ticket"),
    ("[Ertaga](date) [kechqurun](time) [Andijonga](to) bormoqchiman", "buy_ticket"),
    ("[Nukusga](to) poyezd soat nechada joʻnaydi?", "schedule"),
    ("[Ertaga](date) [Buxorodan](from) poyezdlar jadvali", "schedule"),
    ("[Shanba kuni](date) [ikki](count) kishilik stol, iltimos", "book_table"),
    ("Soat [19:30](time) ga stol band qilsa boʻladimi?", "book_table"),
    ("Salom, [bugun](date) [kechqurun](time) stol kerak", "book_table"),
    ("Fikrimni oʻzgartirdim, bronni bekor qiling", "cancel"),
    ("Poyezd chiptasini qaytarish", "cancel"),
    ("Xayrli kun, yordam bering", "greeting"),
    ("Katta rahmat, xayr", "thanks_bye"),
]


import re
import numpy as np

TOKEN_RE = r"\d+(?::\d+)?(?:-[^\W\d_]+)?|[^\W\d_]+(?:-[^\W\d_]+)*"
MARK_RE = r"\[([^\]]+)\]\((\w+)\)"

def tokenize(text):
    return re.findall(TOKEN_RE, text)

def parse_markup(marked):
    # "[Ташкента](from) ..." -> (tokenlar, BIO teglar)
    tokens, tags, pos = [], [], 0
    for m in re.finditer(MARK_RE, marked):
        for t in tokenize(marked[pos:m.start()]):
            tokens.append(t); tags.append("O")
        for k, t in enumerate(tokenize(m.group(1))):
            tokens.append(t); tags.append(("B-" if k == 0 else "I-") + m.group(2))
        pos = m.end()
    for t in tokenize(marked[pos:]):
        tokens.append(t); tags.append("O")
    return tokens, tags

def plain(marked):
    return re.sub(MARK_RE, r"\1", marked)

train = [(plain(s), intent) for intent, sents in train_data.items() for s in sents]
test = [(plain(s), intent) for s, intent in test_data]
train_bio = [parse_markup(s) for sents in train_data.values() for s in sents]
test_bio = [parse_markup(s) for s, _ in test_data]
INTENTS = sorted(train_data)

print(len(train), "ta oʻquv gapi,", len(test), "ta test gapi,", "intentlar:", INTENTS)
example = test_data[1][0]
print("\n" + example)
for tok, tag in zip(*parse_markup(example)):
    print(f"   {tok:<14} {tag}")

## 2. Intentlarni tasniflash

### Noldan: softmax-regressiya

Gap → «soʻzlar qopi» vektori $x$ (soʻz gapda boʻlsa 1). Model: $P(c\mid x)=\text{softmax}(xW+b)_c$. Kross-entropiya boʻyicha gradient tushish bilan oʻrgatamiz; logitlar boʻyicha gradient oddiy: $P-Y$, bu yerda $Y$ — toʻgʻri intentning one-hot vektori. Oʻqitishda boʻlmagan soʻzlarni model umuman koʻrmaydi.

In [ ]:
def bag(text):
    return sorted(set(w.lower() for w in tokenize(text)))

vocab = sorted({w for s, _ in train for w in bag(s)})
w2i = {w: i for i, w in enumerate(vocab)}

def featurize(texts):
    X = np.zeros((len(texts), len(vocab)))
    for r, s in enumerate(texts):
        for w in bag(s):
            if w in w2i:                 # notanish soʻzlarni tashlab ketamiz
                X[r, w2i[w]] = 1.0
    return X

def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

X_tr = featurize([s for s, _ in train])
y_tr = np.array([INTENTS.index(c) for _, c in train])
X_te = featurize([s for s, _ in test])
y_te = np.array([INTENTS.index(c) for _, c in test])
Y = np.eye(len(INTENTS))[y_tr]                      # one-hot

rng = np.random.default_rng(0)
W = rng.normal(0, 0.01, size=(len(vocab), len(INTENTS)))
b = np.zeros(len(INTENTS))
lr, lam = 0.5, 1e-3
for epoch in range(301):
    P = softmax(X_tr @ W + b)
    loss = -np.mean(np.log(P[np.arange(len(y_tr)), y_tr])) + lam / 2 * (W ** 2).sum()
    G = (P - Y) / len(y_tr)                          # logitlar boʻyicha kross-entropiya gradienti
    W -= lr * (X_tr.T @ G + lam * W)
    b -= lr * G.sum(axis=0)
    if epoch % 100 == 0:
        print(f"epoxa {epoch:3d}  loss = {loss:.3f}")

P_te = softmax(X_te @ W + b)
print(f"\naniqlik (noldan, soʻzlar): {np.mean(P_te.argmax(1) == y_te):.2f}\n")
for (s, gold), p in zip(test, P_te):
    k = p.argmax()
    mark = "" if INTENTS[k] == gold else "   <-- " + gold
    print(f"{s:<52} {INTENTS[k]:<11} {p[k]:.2f}{mark}")

### scikit-learn: oʻsha soʻzlar va belgi n-grammalari

Oʻsha binar belgilardagi `LogisticRegression` — xuddi shu model, faqat boshqa optimizator va boshqa regulyarizatsiya kuchi bilan, shuning uchun bahsli gaplarda javoblar farq qilishi mumkin. Ikkinchi model belgi n-grammalaridan foydalanadi (soʻz ichida 2–4 belgi). Oʻzbek tilida bu ayniqsa muhim: «bekor qiling», «bekor qilmoqchiman», «bekor qilish» — bir xil oʻzak, turli qoʻshimchalar; «chiptani», «chiptasini», «chipta» umumiy n-grammalarga ega, shuning uchun notanish shakl ham taniladi.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

X_text_tr = [s for s, _ in train]
y_text_tr = [c for _, c in train]
X_text_te = [s for s, _ in test]
y_text_te = [c for _, c in test]

models = {
    "soʻzlar (LogisticRegression)": make_pipeline(
        CountVectorizer(tokenizer=bag, token_pattern=None, lowercase=False, binary=True),
        LogisticRegression(C=10, max_iter=2000)),
    "belgi n-grammalari (LogisticRegression)": make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True),
        LogisticRegression(C=10, max_iter=2000)),
}
pred = {}
for name, m in models.items():
    m.fit(X_text_tr, y_text_tr)
    pred[name] = m.predict(X_text_te)
    print(f"{name:<34} aniqlik = {accuracy_score(y_text_te, pred[name]):.2f}")

print()
for s, gold, a, c in zip(X_text_te, y_text_te, pred["soʻzlar (LogisticRegression)"], pred["belgi n-grammalari (LogisticRegression)"]):
    print(f"{s:<52} {gold:<11} {a:<11} {c:<11}")

intent_clf = models["belgi n-grammalari (LogisticRegression)"]
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_text_te, pred["belgi n-grammalari (LogisticRegression)"], labels=INTENTS,
                                        ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title("Intentlar: belgi n-grammalari")
ax.set_xlabel("bashorat")
ax.set_ylabel("toʻgʻrisi")
plt.tight_layout()
plt.show()

## 3. Slotlarni toʻldirish

### Qoidalar

Eng oddiy slot toʻldiruvchi — lugʻatlar va shablonlar: shaharlar roʻyxati (gazetteer), sana va vaqt soʻzlari, sonlar. Oʻzbek tilida shaharning **roli** (qayerdan yoki qayerga) kelishik qoʻshimchasi bilan ifodalanadi: «Toshkent**dan**» → `from`, «Samarqand**ga**», «Buxoro**gacha**» → `to`. Rus tilida esa bu vazifani predlog bajaradi («из Ташкента», «в Самарканд»). Shuning uchun oʻzbekcha qoidalar soʻzni oʻzak va qoʻshimchaga ajratadi.


Baholash **spanlar** boʻyicha: slot faqat turi ham, chegaralari ham mos kelsa hisobga olinadi (NER dagi kabi). Quyida qoidalarning test gaplaridagi bahosi va xatolari: masalan, gazetteerda yoʻq «Andijon» shahrini qoidalar bilmaydi.

In [ ]:
# shaharlar gazetteri: oʻzak -> shahar nomi. Shaharning roli kelishik qoʻshimchasidan aniqlanadi!
CITIES = {"toshkent": "Toshkent", "samarqand": "Samarqand", "buxoro": "Buxoro", "xiva": "Xiva",
          "nukus": "Nukus", "qarshi": "Qarshi", "termiz": "Termiz"}
FROM_SUF = ("dan",)                       # chiqish kelishigi: Toshkentdan
TO_SUF = ("ga", "ka", "qa", "gacha")      # joʻnalish kelishigi: Samarqandga, Buxorogacha
DATE_WORDS = {"bugun", "ertaga", "indin", "indinga"}
WEEKDAYS = {"dushanba", "seshanba", "chorshanba", "payshanba", "juma", "shanba", "yakshanba"}
TIME_WORDS = {"ertalab", "ertalabki", "kechqurun", "tushda", "kechasi"}
COUNT_WORDS = {"bir": 1, "bitta": 1, "ikki": 2, "ikkita": 2, "uch": 3, "uchta": 3, "toʻrt": 4,
               "toʻrtta": 4, "besh": 5, "beshta": 5, "olti": 6, "oltita": 6}

def split_city(tok):
    # "Samarqandga" -> ("Samarqand", "ga")
    low = tok.lower()
    for stem_, name in CITIES.items():
        if low.startswith(stem_):
            return name, low[len(stem_):]
    return None, None

def rule_tags(toks):
    tags = ["O"] * len(toks)
    low = [t.lower() for t in toks]
    unassigned = []
    for i, w in enumerate(low):
        city, suf = split_city(w)
        if re.fullmatch(r"\d{1,2}:\d{2}", w) or w in TIME_WORDS:
            tags[i] = "B-time"
        elif re.fullmatch(r"\d{1,2}-[^\W\d_]+", w) or w in DATE_WORDS:
            tags[i] = "B-date"                                   # «10-may», «ertaga»
        elif w in WEEKDAYS:
            tags[i] = "B-date"
            if i + 1 < len(toks) and low[i + 1] in ("kuni", "kungi"):
                tags[i + 1] = "I-date"                           # «juma kuni»
        elif w in COUNT_WORDS:
            tags[i] = "B-count"
        elif city:
            if suf in FROM_SUF:
                tags[i] = "B-from"
            elif suf in TO_SUF:
                tags[i] = "B-to"
            elif suf == "":
                unassigned.append(i)
    # qoʻshimchasiz shaharlar («Toshkent – Nukus»): birinchisi — qayerdan, ikkinchisi — qayerga
    for k, i in enumerate(unassigned):
        tags[i] = "B-from" if k == 0 and len(unassigned) > 1 else "B-to"
    return tags

def normalize(slot, value):
    # slot qiymatini kanonik koʻrinishga keltiramiz
    if slot in ("from", "to"):
        return split_city(value)[0] or value
    if slot == "count":
        return COUNT_WORDS.get(value.lower(), int(value) if value.isdigit() else value)
    return value.lower()

for s in ["Juma kuni kechqurun Samarqanddan Buxorogacha ikkita chipta kerak", "Toshkent – Nukus, 10-may, soat 8:00"]:
    toks = tokenize(s)
    print(" ".join(f"{t}/{x}" if x != "O" else t for t, x in zip(toks, rule_tags(toks))))


def spans(tags):
    # BIO teglar -> spanlar toʻplami (slot, boshi, oxiri)
    out, cur = [], None
    for i, t in enumerate(tags + ["O"]):
        if cur and not (t.startswith("I-") and t[2:] == cur[0]):
            out.append((cur[0], cur[1], i)); cur = None
        if t.startswith("B-") or (t.startswith("I-") and cur is None):
            cur = (t[2:], i)
    return set(out)

def span_prf(gold_tags, pred_tags):
    tp = fp = fn = 0
    for g, p in zip(gold_tags, pred_tags):
        G, P = spans(list(g)), spans(list(p))
        tp += len(G & P); fp += len(P - G); fn += len(G - P)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return prec, rec, f1

rule_pred = [rule_tags(toks) for toks, _ in test_bio]
gold = [tags for _, tags in test_bio]
print("qoidalar: P = %.2f  R = %.2f  F1 = %.2f" % span_prf(gold, rule_pred))
for (toks, g), p in zip(test_bio, rule_pred):
    if spans(g) != spans(p):
        print("  ", " ".join(f"{t}/{x}" if x != "O" else t for t, x in zip(toks, p)))
        print("   toʻgʻrisi:", " ".join(f"{t}/{x}" if x != "O" else t for t, x in zip(toks, g)))

### CRF: belgilashdan oʻrganadigan slot toʻldiruvchi

Chiziqli zanjirli CRF (9-mavzu) teglar ketma-ketligini birdaniga baholaydi: token belgilari (soʻzning oʻzi, oxirgi harflari, bosh harf, qoʻshni soʻzlar) va teglar orasidagi oʻtish vaznlari: masalan, `O → I-date` oʻtishi (boshlanmagan sananing davomi) jarima oladi. Oʻqitish — oʻsha 60 ta belgilangan gapda. Quyida testdagi sifat, keyin har bir teg uchun eng kuchli belgilar hamda eng «taqiqlangan» va «ruxsat etilgan» oʻtishlar.

In [ ]:
import sklearn_crfsuite

def word_features(toks, i):
    w = toks[i]
    f = {
        "bias": 1.0,
        "w": w.lower(),
        "suf2": w.lower()[-2:],
        "suf3": w.lower()[-3:],
        "pre4": w.lower()[:4],
        "title": w[:1].isupper(),
        "first": i == 0,
        "digit": w[:1].isdigit(),
        "has_colon": ":" in w,
    }
    f["prev"] = toks[i - 1].lower() if i > 0 else "<s>"
    f["next"] = toks[i + 1].lower() if i + 1 < len(toks) else "</s>"
    return f

def sent_features(toks):
    return [word_features(toks, i) for i in range(len(toks))]

crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.05, c2=0.05, max_iterations=200,
                           all_possible_transitions=True)
crf.fit([sent_features(t) for t, _ in train_bio], [tags for _, tags in train_bio])

crf_pred = [list(p) for p in crf.predict([sent_features(t) for t, _ in test_bio])]
print("CRF:     P = %.2f  R = %.2f  F1 = %.2f" % span_prf(gold, crf_pred))
print("qoidalar: P = %.2f  R = %.2f  F1 = %.2f\n" % span_prf(gold, rule_pred))
for (toks, g), p, r in zip(test_bio, crf_pred, rule_pred):
    print(" ".join(f"{t}/{x}" if x != "O" else t for t, x in zip(toks, p)),
          "" if spans(g) == spans(p) else "   <-- CRF xato qildi")


from collections import Counter
for label in ["B-from", "B-to", "B-date", "B-count"]:
    feats = [(f, w) for (f, l), w in crf.state_features_.items() if l == label]
    top = sorted(feats, key=lambda t: -t[1])[:6]
    print(f"{label:<8}", ", ".join(f"{f} ({w:.1f})" for f, w in top))

print("\nTeglar orasidagi oʻtishlar (eng manfiy va eng musbat vaznlar):")
trans = sorted(crf.transition_features_.items(), key=lambda t: t[1])
for (a, b_), w in trans[:3] + trans[-3:]:
    print(f"   {a:>7} -> {b_:<7} {w:6.2f}")

CRF nimani oʻrgandi? `B-from` va `B-to` uchun model soʻz oxiriga (`suf3:dan`, `suf2:ga`) tayanadi — aynan shu kelishik qoʻshimchalarini biz qoidalarga qoʻlda yozgan edik. Rus tilidagi variantda esa eng kuchli belgilar oldingi soʻz-predlog boʻladi.

Ikki yondashuv xatolarini solishtiring. CRF notanish «Andijonga» shahrini qoʻshimcha (-ga) va bosh harf boʻyicha topdi, qoidalar esa yoʻq — u gazetteerda yoʻq. Lekin qoidalar lugʻatdan «bitta» soʻzini biladi, u 60 ta oʻquv gapida yoʻq edi va CRF uni oʻtkazib yubordi. «Salom, bugun kechqurun…» gapida CRF sanani ham, vaqtni ham topmadi: kichik korpusda u kontekstga (gap boshi, qoʻshni soʻzlar) haddan tashqari tayanadi, oʻqitishda esa «Bugun» faqat gap boshida uchragan. Odatda kombinatsiya yutadi: «soʻz gazetteerda bor» belgisi CRF ga qoʻshiladi.


### Tushunish moduli (NLU) toʻliq

Hammasini birlashtiramiz: intent — scikit-learn belgi modeli, slotlar — qoidalar (yoki CRF), slot qiymatlari normallashtiriladi: «Buxorodan» → «Buxoro», «ikkita» → 2.

In [ ]:
def extract_slots(toks, tags):
    # BIO -> [(slot, normallangan qiymat)]
    out = []
    for name, i, j in sorted(spans(tags), key=lambda s: s[1]):
        out.append((name, normalize(name, " ".join(toks[i:j]))))
    return out

def nlu(text, tagger="rules"):
    toks = tokenize(text)
    proba = intent_clf.predict_proba([text])[0]
    k = int(proba.argmax())
    tags = rule_tags(toks) if tagger == "rules" else list(crf.predict([sent_features(toks)])[0])
    return str(intent_clf.classes_[k]), round(float(proba[k]), 2), extract_slots(toks, tags)

for s in ["Juma kuni Buxorodan Toshkentga ikkita chipta kerak", "Bugun soat 19:30 ga ikki kishilik stol", "Iltimos, bekor qiling"]:
    print(s)
    print("   ", nlu(s), "\n")

## 4. Freymli dialog menejeri

**Freym** — vazifani bajarish uchun toʻldirilishi kerak boʻlgan slotlar toʻplami: chipta uchun `from, to, date, count`, stol uchun `date, time, count`. Menejer — chekli avtomat: holatlar `START → ASK_slot → … → CONFIRM → DONE`. Har qadamda u freymni gapdagi slotlar bilan yangilaydi va birinchi toʻldirilmagan slotni soʻraydi. `CONFIRM` holatida «ha» buyurtmani yakunlaydi, yangi qiymatli «yoʻq» esa slotni tuzatadi. `cancel` intenti istalgan holatda freymni tozalaydi.

**Shablon asosidagi generatsiya (NLG)** menejer harakatini matnga aylantiradi. Oʻzbek shablonida qoʻshimcha qoʻshiladi: «Toshkent**dan** Buxoro**ga**». Son bilan ot esa moslashmaydi: «2 ta chipta», «5 kishi» — rus tilidagi «2 билета / 5 билетов» kabi shakllar kerak emas.

In [ ]:
# Shablon asosida javob generatsiyasi (template NLG). Oʻzbek shablonida kelishik qoʻshimchasi qoʻshiladi,
# son bilan otning moslashuvi esa kerak emas: «2 ta chipta», «5 kishi» (ot birlikda qoladi).
YES = {"ha", "toʻgʻri", "tasdiqlayman", "albatta"}
NO = {"yoʻq", "notoʻgʻri"}

def add_suffix(word, suf):
    # -ga/-ka/-qa: k bilan tugasa -ka, q bilan tugasa -qa (Termiz -> Termizga, Qoʻqon -> Qoʻqonga)
    if suf == "ga" and word.endswith("k"):
        suf = "ka"
    elif suf == "ga" and word.endswith("q"):
        suf = "qa"
    return word + suf

def at_time(t):
    return "soat " + t + " da" if re.fullmatch(r"\d{1,2}:\d{2}", t) else t   # «soat 19:00 da» / «kechqurun»

ASK = {
    ("buy_ticket", "from"): "Qayerdan joʻnaysiz?",
    ("buy_ticket", "to"): "Qayerga borasiz?",
    ("buy_ticket", "date"): "Chipta qaysi kunga kerak?",
    ("buy_ticket", "count"): "Nechta chipta kerak?",
    ("book_table", "date"): "Stolni qaysi kunga band qilay?",
    ("book_table", "time"): "Soat nechaga?",
    ("book_table", "count"): "Necha kishi boʻlasiz?",
}

def describe(task, f):
    if task == "buy_ticket":
        return (f"{add_suffix(f['from'], 'dan')} {add_suffix(f['to'], 'ga')} {f['count']} ta chipta, "
                f"sana: {f['date']}")
    return f"{f['count']} kishilik stol, sana: {f['date']}, vaqt: {at_time(f['time'])}"

def nlg(action, task=None, arg=None):
    if action == "ask":
        return ASK[(task, arg)]
    if action == "confirm":
        return f"Demak: {describe(task, arg)}. Toʻgʻrimi?"
    if action == "done":
        return f"Tayyor! Rasmiylashtirildi: {describe(task, arg)}."
    return {
        "greet": "Assalomu alaykum! Men poyezd chiptasini sotishim yoki stol band qilishim mumkin.",
        "what_change": "Nimani oʻzgartirish kerak?",
        "cancelled": "Xoʻp, buyurtma bekor qilindi.",
        "bye": "Xayr, salomat boʻling!",
        "fallback": "Kechirasiz, men faqat chipta sotish va stol band qilishni bilaman.",
    }[action]

print(nlg("confirm", "buy_ticket", {"from": "Buxoro", "to": "Xiva", "date": "juma kuni", "count": 1}))
print(nlg("confirm", "buy_ticket", {"from": "Toshkent", "to": "Termiz", "date": "ertaga", "count": 3}))
print(nlg("confirm", "book_table", {"date": "shanba kuni", "time": "kechqurun", "count": 5}))

print()

dialogue_1 = [
    "Assalomu alaykum",
    "Ertaga Samarqandga chipta olmoqchiman",
    "Toshkentdan",
    "Ikkita chipta",
    "Yoʻq, Buxoroga",
    "Ha",
    "Rahmat, xayr",
]
dialogue_2 = [
    "Uch kishilik stol kerak",
    "Shanba kuni",
    "Soat 19:00 da",
    "Bekor qiling, rejalar oʻzgardi",
]


REQUIRED = {"buy_ticket": ["from", "to", "date", "count"],
            "book_table": ["date", "time", "count"]}

class FrameDM:
    # freymli menejer: holat, faol vazifa, toʻldirilgan slotlar
    def __init__(self, nlu):
        self.nlu = nlu
        self.reset()
        self.path = ["START"]

    def reset(self):
        self.task, self.frame, self.state = None, {}, "START"

    def go(self, state):
        self.state = state
        self.path.append(state)

    def fill(self, slots):
        for name, value in slots:
            if name in REQUIRED[self.task]:
                self.frame[name] = value

    def next_question(self):
        missing = [s for s in REQUIRED[self.task] if s not in self.frame]
        if missing:
            self.go("ASK_" + missing[0])
            return nlg("ask", self.task, missing[0])
        self.go("CONFIRM")
        return nlg("confirm", self.task, self.frame)

    def respond(self, text):
        intent, conf, slots = self.nlu(text)
        words = {w.lower() for w in tokenize(text)}
        if self.state == "CONFIRM":                       # tasdiqni kutyapmiz: «ha» -> DONE, «yoʻq» + yangi slot -> tuzatish
            if words & YES:
                self.go("DONE")
                return nlg("done", self.task, self.frame)
            if words & NO or slots:
                self.fill(slots)
                return nlg("what_change") if not slots else self.next_question()
        if intent == "cancel" and conf > 0.4:
            self.reset(); self.go("START")
            return nlg("cancelled")
        if intent == "thanks_bye" and not slots:
            self.go("END")
            return nlg("bye")
        if self.task is None or self.state == "DONE":
            if intent in REQUIRED:
                self.reset(); self.task = intent
            elif intent == "greeting":
                return nlg("greet")
            else:
                return nlg("fallback")
        self.fill(slots)
        return self.next_question()

def run_dialogue(user_turns, nlu_fn=nlu):
    dm = FrameDM(nlu_fn)
    for u in user_turns:
        reply = dm.respond(u)
        print(f"Mijoz:   {u}")
        print(f"Bot:    {reply}")
        print(f"          [{dm.state}] {dm.task} {dm.frame}\n")
    return dm

dm1 = run_dialogue(dialogue_1)

Ikkinchi dialog: foydalanuvchi bekor qiladigan stol bandi. Keyin avtomatning bosib oʻtilgan holatlarini chizamiz.

In [ ]:
dm2 = run_dialogue(dialogue_2)


import networkx as nx

pos = {"START": (0, 0), "ASK_from": (1, 1.5), "ASK_to": (2, 1.5), "ASK_date": (3, 1.5),
       "ASK_time": (3, -1.5), "ASK_count": (4, 0), "CONFIRM": (5, 0), "DONE": (6, 0.8), "END": (7, 0)}
G = nx.DiGraph()
G.add_nodes_from(pos)
edges1 = list(zip(dm1.path, dm1.path[1:]))
edges2 = list(zip(dm2.path, dm2.path[1:]))
G.add_edges_from(edges1 + edges2)
fig, ax = plt.subplots(figsize=(11, 4))
visited = set(dm1.path) | set(dm2.path)
nx.draw_networkx_nodes(G, pos, node_size=1900, ax=ax,
                       node_color=["#F2C14E" if n in visited else "#DDDDDD" for n in G])
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
for edges, color, rad in [(edges1, "#2B55C9", 0.15), (edges2, "#D9822B", -0.15)]:
    e = [x for x in dict.fromkeys(edges) if x[0] != x[1]]
    nx.draw_networkx_edges(G, pos, edgelist=e, edge_color=color, width=2, ax=ax, node_size=1900,
                           arrowsize=16, connectionstyle=f"arc3,rad={rad}")
ax.plot([], [], color="#2B55C9", label="1-dialog (chipta)")
ax.plot([], [], color="#D9822B", label="2-dialog (stol, bekor qilish)")
ax.legend(loc="lower left")
ax.set_title("Ikki dialogda bosib oʻtilgan holatlar")
ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Ssenariy va generatsiya

Ssenariyli tizim toʻliq boshqariladi: u faqat freymda bor narsani aytadi, lekin bir xil eshitiladi. Generativ model javobni oʻzi yozadi. Muammoni aniq shaharlar va sanalar yozilgan 15 ta tizim javobida oʻrgatilgan oʻyinchoq bigramm modelda koʻrsatamiz (uchta javobda sana tilga olinmagan) (harorat $T$ taqsimotni oʻzgartiradi, 16-mavzudagi kabi).

1. **Oddiy generatsiya**: model freymni koʻrmaydi va oʻqitishda tez-tez uchragan shaharlarni qoʻyadi — slot qiymatlarini «toʻqib chiqaradi» (gallyutsinatsiya).
2. **Deleksikalizatsiya**: oʻquv javoblarida qiymatlar `FROM`, `TO`, `DATE` belgilari bilan almashtirilgan. Model shablon yaratadi, qiymatlar esa freymdan kerakli qoʻshimcha bilan qoʻyiladi. Notoʻgʻri qiymat boʻlmaydi, lekin model slotni tushirib qoldirishi mumkin.

Bu oʻyinchoq, haqiqiy model emas: u faqat «ravonlik va nazorat» oʻrtasidagi murosani koʻrsatadi.

In [ ]:
# aniq slot qiymatlari bilan tizim javoblari — oʻyinchoq generativ modelning «oʻquv korpusi»
responses = [
    "Chiptangiz tayyor: ertaga Toshkentdan Samarqandga.",
    "Chipta rasmiylashtirildi, siz bugun Buxorodan Toshkentga ketasiz.",
    "Tayyor, siz indinga Samarqanddan Buxoroga ketasiz.",
    "Xoʻp, chipta tayyor: ertaga Toshkentdan Xivaga, oq yoʻl!",
    "Chiptangiz rasmiylashtirildi: bugun Nukusdan Toshkentga, oq yoʻl!",
    "Siz ertaga Toshkentdan Buxoroga ketasiz, chipta tayyor.",
    "Ajoyib, siz indinga Xivadan Samarqandga ketasiz.",
    "Chipta tayyor: bugun Samarqanddan Toshkentga.",
    "Tayyor, chiptangiz rasmiylashtirildi: ertaga Buxorodan Nukusga.",
    "Siz indinga Toshkentdan Nukusga ketasiz, oq yoʻl!",
    "Xoʻp, siz bugun Xivadan Toshkentga ketasiz.",
    "Chiptangiz tayyor: ertaga Samarqanddan Xivaga, oq yoʻl!",
    "Chiptangiz tayyor: Buxorodan Toshkentga.",
    "Siz Samarqanddan Xivaga ketasiz, oq yoʻl!",
    "Tayyor, chipta rasmiylashtirildi: Toshkentdan Nukusga.",
]
# soʻz shakllari -> (slot, asosiy shakl); rol kelishik qoʻshimchasi bilan ifodalanadi: -dan / -ga
CITY_LIST = ["Toshkent", "Samarqand", "Buxoro", "Xiva", "Nukus"]
SLOT_FORMS = {
    "from": {add_suffix(c, "dan"): c for c in CITY_LIST},
    "to": {add_suffix(c, "ga"): c for c in CITY_LIST},
    "date": {w: w for w in ["ertaga", "bugun", "indinga"]},
}

def delexicalize(s):
    # slot qiymatlarini belgilar bilan almashtiramiz: «Toshkentdan» -> «FROM»
    out = []
    for t in tokenize(s):
        for name in ["from", "to", "date"]:
            if t in SLOT_FORMS[name]:
                t = name.upper(); break
        out.append(t)
    return " ".join(out)

def lexicalize(s, frame):
    # teskari amal: belgilar -> freymdagi qiymat + kerakli qoʻshimcha
    return (s.replace("FROM", add_suffix(frame["from"], "dan")).replace("TO", add_suffix(frame["to"], "ga"))
             .replace("DATE", frame["date"]))

frame_demo = {"from": "Toshkent", "to": "Buxoro", "date": "ertaga", "count": 2}
print(len(responses), "ta javob")


from collections import defaultdict

def train_bigram(sentences):
    counts = defaultdict(lambda: defaultdict(int))
    for s in sentences:
        toks = ["<s>"] + tokenize(s) + ["</s>"]
        for a, b_ in zip(toks, toks[1:]):
            counts[a][b_] += 1
    return counts

def sample(counts, T=1.0, rng=None, max_len=25):
    out, cur = [], "<s>"
    while len(out) < max_len:
        words = list(counts[cur])
        logits = np.log(np.array([counts[cur][w] for w in words], dtype=float))
        p = np.exp((logits - logits.max()) / T)
        p /= p.sum()
        cur = words[rng.choice(len(words), p=p)]
        if cur == "</s>":
            break
        out.append(cur)
    return " ".join(out)

def mentioned(text):
    # javobda qaysi slot qiymatlari tilga olingan
    found = {}
    for t in tokenize(text):
        for name in ["from", "to", "date"]:
            if t in SLOT_FORMS[name]:
                found.setdefault(name, []).append(SLOT_FORMS[name][t])
    return found

def check(text, frame):
    m = mentioned(text)
    wrong = any(v != frame[k] for k, vs in m.items() for v in vs)
    missing = any(k not in m for k in ["from", "to", "date"])
    return wrong, missing

raw_lm = train_bigram(responses)
delex_lm = train_bigram([delexicalize(r) for r in responses])
print("deleksikalizatsiyalangan javob:", delexicalize(responses[0]))
print()

rng = np.random.default_rng(1)
print("freym:", frame_demo)
for T in [0.5, 1.0, 2.0]:
    print(f"\nT = {T}")
    for _ in range(3):
        print("   oddiy:  ", sample(raw_lm, T, rng))
    for _ in range(2):
        print("   delek.:", lexicalize(sample(delex_lm, T, rng), frame_demo))

Har bir harorat uchun 300 tadan javob hisoblaymiz: slot qiymati **notoʻgʻri** boʻlgan javoblar ulushi va qaysidir slot **tushib qolgan** javoblar ulushi.

In [ ]:
rows = []
for T in [0.5, 1.0, 1.5, 2.0]:
    r = np.array([check(sample(raw_lm, T, rng), frame_demo) for _ in range(300)])
    d = np.array([check(lexicalize(sample(delex_lm, T, rng), frame_demo), frame_demo) for _ in range(300)])
    rows.append((T, r[:, 0].mean(), r[:, 1].mean(), d[:, 0].mean(), d[:, 1].mean()))
    print(f"T={T}:  oddiy notoʻgʻri qiymat {r[:, 0].mean():.0%}, slot tushib qolgan {r[:, 1].mean():.0%}   |   "
          f"delek. notoʻgʻri qiymat {d[:, 0].mean():.0%}, slot tushib qolgan {d[:, 1].mean():.0%}")
tpl = nlg("done", "buy_ticket", frame_demo)
print(f"\nshablon: {tpl}  ->  notoʻgʻri qiymat/slot tushib qolgan: {check(tpl, frame_demo)}")

rows = np.array(rows)
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(rows[:, 0], rows[:, 1], "o-", label="oddiy: notoʻgʻri qiymat")
ax.plot(rows[:, 0], rows[:, 4], "s--", label="delek.: slot tushib qolgan")
ax.plot(rows[:, 0], rows[:, 3], "^-", label="delek.: notoʻgʻri qiymat")
ax.set_xlabel("harorat T")
ax.set_ylim(-0.03, 1.03)
ax.set_title("Generativ javoblar: harorat boʻyicha xatolar")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Haqiqiy modelda

Zamonaviy tizimlar koʻpincha javobni katta til modeli bilan generatsiya qiladi va freymni unga soʻrov (prompt) ichida beradi. Quyida `Qwen/Qwen2.5-0.5B-Instruct` (0,5 mlrd parametr, ≈1 GB) JSON dagi freymni oladi va mijozga javob yozadi. Kichik modellar oʻzbek tilida ancha zaif: grammatik xatolar, rus yoki ingliz soʻzlari aralashishi mumkin. Freymdagi barcha qiymatlar javobga tushganmi va model ortiqcha narsa (narx, vaqt, poyezd raqami) qoʻshmaganmi — tekshiring. Taqqoslash uchun tizim soʻrovini rus yoki ingliz tilida yozib koʻring. Katak faqat internetli Colab da ishlaydi; CPU da generatsiya bir daqiqacha davom etadi.

In [ ]:
# Colab da transformers va torch allaqachon oʻrnatilgan
try:
    import json, torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    name = "Qwen/Qwen2.5-0.5B-Instruct"                # ≈1 GB, bir marta yuklanadi
    tok = AutoTokenizer.from_pretrained(name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = "auto" if device == "cuda" else torch.float32     # CPU da float32 bfloat16 dan tezroq
    llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(device)
    messages = [
        {"role": "system", "content": "Sen temir yoʻl kassasining xushmuomala yordamchisisan. Mijozga chipta rasmiylashtirilganini oʻzbek tilida bitta gap bilan ayt. Faqat freymdagi maʼlumotlardan foydalan."},
        {"role": "user", "content": "Buyurtma freymi: " + json.dumps(frame_demo, ensure_ascii=False)},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok([text], return_tensors="pt").to(device)
    torch.manual_seed(0)
    for k in range(4):
        out = llm.generate(**inputs, max_new_tokens=60, do_sample=True, temperature=0.9, top_p=0.95)
        reply = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        print(f"{k + 1}. {reply}")
        print("   tilga olingan slot qiymatlari:", mentioned(reply), "\n")
except Exception as e:
    print("Modelni ishga tushirib boʻlmadi (Colab da internet kerak):", e)

## 6. Topshiriqlar

1. **Yangi intent.** Slot belgilari bilan 8–10 ta gapdan iborat `price` intentini qoʻshing («Xivagacha chipta qancha turadi?»). Ikkala scikit-learn modelini qayta oʻrgating. Boshqa intentlardagi qaysi gaplar endi `price` bilan adashtiriladi?
2. **Yangi slot.** 5–6 ta `buy_ticket` gapiga `class` slotini qoʻshing (ekonom, biznes), CRF ni qayta oʻrgating va uni «Ertaga Samarqandga biznes klass chipta» gapida tekshiring.
3. **CRF belgilarini olib tashlash (ablation).** `word_features` dan avval `suf2`/`suf3`, keyin `prev`/`next` belgilarini olib tashlang. Testdagi F1 qanday oʻzgaradi? Oʻzbek tili uchun qaysi belgilar muhimroq va nega (maslahat: kelishik qoʻshimchalari)?
4. **Menejer.** Hozir «Qayerga borasiz?» savoliga qoʻshimchasiz «Samarqand» javobini qoidalar tasodifan `to` deb belgilaydi. Menejer kutilayotgan slotni (`self.state`) NLU ga uzatsin va qoʻshimchasiz shahar soʻralgan slotga yozilsin. «Qayerdan joʻnaysiz?» savoliga foydalanuvchi «Toshkent» deb javob beradigan dialogda tekshiring.
5. **Tekshiruvli generatsiya.** Deleksikalizatsiyalangan javob generatsiya qiladigan, unda `FROM`, `TO`, `DATE` belgilarining uchalasi ham borligini tekshiradigan va boʻlmasa qayta generatsiya qiladigan (koʻpi bilan 20 urinish) funksiya yozing. $T = 1$ va $T = 2$ da oʻrtacha nechta urinish kerak? Nega bunday filtrni oddiy (deleksikalizatsiyalanmagan) modelga freymni bilmasdan qoʻllab boʻlmaydi?

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing